# Imports

In [ ]:
from collections import Counter
from collections import defaultdict
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from openpyxl.utils.dataframe import dataframe_to_rows
from operator import sub
from sklearn.preprocessing import LabelEncoder
from pathlib import Path
import types
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import contextlib
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
from transformers import AutoModel, AutoTokenizer
import gc
import h5py  # For efficient storage of large arrays
import hashlib
import json
import matplotlib.pyplot as plt
import numpy as np
import time
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
import openpyxl
import os
import pandas as pd
import pickle
import random
import re
import seaborn as sns
import string
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import shutil
import xlsxwriter
from scipy.sparse import csr_matrix
from captum.attr import IntegratedGradients, LayerConductance, LayerActivation
from captum.attr import visualization as viz

In [ ]:
from config import get_model_config, COMPLETION_LABELS
from data.load_data import mount_drive, load_data
from data.process_data import filter_proceedings, remove_multi_defendant_cases, create_combined_dockets, filter_to_magistrate_started_cases, remove_extradition_cases
from data.sequence_builder import create_enhanced_training_sequences_cached
from regex_labels.label_generator_and_cache_manager import generate_and_cache_regex_labels
from models.transformer_creation import create_model_from_data
from training.trainer import train_case_transformer_with_mlb
from training.label_processor import EnhancedLabelProcessor
from training.evaluation import evaluate_with_teacher_forcing
from analysis.cooccurrence import export_cooccurrence_from_valid_completed_cases, create_cooccurrence_matrix_excel
from analysis.frequency_analysis import analyze_raw_label_frequencies_combined, analyze_raw_label_frequencies_singular, print_label_distribution_by_court
from analysis.diagnostic import comprehensive_training_diagnostic
from models.baseline_model import run_baseline
from utils.cache_directory_utils import CacheDirectoryManager
from utils.model_memory_utils import analyze_model_memory, debug_oom_sample
from models.embeddings import extract_and_cache_bert_embeddings_streaming
from analysis.byclerk import analyze_clerk_data
from models.interpretability import CaseTransformerInterpretability, CaptumWrapper
from models.most_confident_wrong_answer_analyzer import run_confident_misclassification_analysis
from data.k_fold_methods import KFoldCVRunner
from data.dataset_creation import sample_from_the_longest_cases, export_long_cases_to_xlsx


In [ ]:
# Set device globally
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

In [ ]:
from notebook_helpers import (
    # Data filtering helpers
    filter_to_completed_cases,
    build_case_counts_df,
    filter_zero_label_proceedings,
    generate_cooccurrence_matrix_excel,
    
    # Data transformation helpers
    transform_labels_to_binary,
    convert_data_to_format,
    
    # Sequence processing helpers
    filter_sequences_with_meaningful_labels,
    split_train_validation,
    prepare_sequences_and_labels,
    
    # Case info extraction
    extract_case_info_from_sequence,
    
    # Prediction helpers
    build_prediction_prefix,
    predict_case_labels,
    predict_step,
    
    # Metrics calculation helpers
    calculate_metrics,
    calculate_case_metrics,
    collect_labels_from_binary,
    count_processed_steps,
    
    # Analysis helpers
    perform_frequency_analysis,
    debug_model_sample,
    
    # Training helpers
    train_model,
    
    # Evaluation helpers
    prepare_validation_data_for_eval,
    evaluate_model,
    print_micro_averaged_metrics,
    print_top_performing_labels,
    
    # Case-level evaluation helpers
    evaluate_case_level,
    print_case_level_metrics,
    save_case_level_results,
    print_best_worst_cases,
    print_f1_distribution,
    calculate_label_metrics_from_cases,
    save_label_metrics_to_excel,
    
    # Advanced analysis helpers
    run_diagnostic_analysis,
    run_baseline_comparison,
    run_confident_misclassification,
    
    # Interpretability helpers
    initialize_interpretability_analyzer,
    select_samples_for_analysis,
    get_model_prediction,
    compute_attributions,
    visualize_attributions,
    get_and_save_feature_summary,
    analyze_single_sample,
    run_attribution_analysis,
    aggregate_component_importance,
    save_and_visualize_aggregate_importance,
    find_sample_with_label,
    analyze_label_specific_features,
    create_label_feature_comparison,
    run_interpretability_analysis,
    
    # Sample prediction helpers
    print_step_prediction_analysis,
    analyze_sample_predictions
)

# Parameters

In [ ]:
parameters_dict = {
    'model_validation_strategy': 'train_val_split',
    'number_of_cases_in_sample': 1000,
    'number_of_epochs_for_training': 20,
    'batch_size_for_training': 16,
    'top_n_performing_labels_to_display': 10,
    'numbers_of_cases_to_analyze_for_interpretability': 200,
    'minimum_confidence_for_wrong_prediction_inclusion': 0.8,
    'number_of_highest_probability_predictions_per_step': 5,
    'maximum_number_of_confident_mistakes_to_return': 25,
    'maximum_validation_sequences_to_process_for_analysis_of_incorrect': 100,
    'num_best_and_worst_cases_to_display': 5,
    'k_fold_num_folds': 5,        # Number of folds for cross-validation
    'k_fold_random_state': 42,    # Random seed for reproducibility
    'k_fold_shuffle': True,       # Shuffle before splitting
    'save_cv_plots': True,        # Whether to save plots of CV metrics
    'cv_plot_filename': 'cross_validation_results.png',
    'save_each_fold_model': True,  # Save every fold’s model state if True
    'save_best_fold_only': False,  # Save only the best fold’s weights if True
    'new_dataset_creation': True
}

# Main Method

## 1. Mount and load data

In [ ]:
drive_root_folder = mount_drive('tulane-courts-shared/v2')
court_data = load_data(drive_root_folder)

In [ ]:
model_configuration = get_model_config()

In [ ]:
court_data = filter_proceedings(court_data)
crawled_original_data_folder_path = Path("/data/krosen5/gdrive-all/tulane-courts-shared/crawled")
court_data, multi_defendant_cases  = remove_multi_defendant_cases(court_data, crawled_original_data_folder_path)

In [ ]:
print("   Creating combined dockets...")
court_data['combined_dockets'] = create_combined_dockets(court_data)

In [ ]:
print("Removing cases that start in CDC and have no MAG proceedings...")
court_data = filter_to_magistrate_started_cases(court_data)

In [ ]:
print("Removing cases that involve some sort of extradition")
court_data = remove_extradition_cases(court_data)
court_data['combined_dockets'] = create_combined_dockets(court_data)


In [ ]:
def clean_the_dictionary(dictionary):
    elements_to_remove = model_configuration.EXCLUDED_LABELS
    cleaned_dict = {}
    
    for key, value in dictionary.items():
        if isinstance(value, list):
            # Remove unwanted elements while preserving order
            cleaned_list = [item for item in value if item not in elements_to_remove]
            # Only keep the key if the list is not empty
            if cleaned_list:
                cleaned_dict[key] = cleaned_list
        else:
            # Keep non-list values as they are
            cleaned_dict[key] = value
    
    return cleaned_dict

In [ ]:
print("2. Processing regex labels...")
labels_cache = get_cache_dir('labels')
cached_outcome_labels = generate_and_cache_regex_labels(court_data, labels_cache, "load_in")
cached_outcome_labels_v2 = clean_the_dictionary(cached_outcome_labels)    
print(f"   Original labels count: {len(cached_outcome_labels)}")
print(f"   Cleaned labels count: {len(cached_outcome_labels_v2)}")

In [ ]:
print("2b. Filtering out proceedings with zero labels...")
court_data, cached_outcome_labels_v2 = filter_zero_label_proceedings(
    court_data, 
    cached_outcome_labels_v2
)
print(f"   Final labels count after filtering: {len(cached_outcome_labels_v2)}")
court_data['combined_dockets'] = create_combined_dockets(court_data)

In [ ]:
print("3. Setting up model...")
computation_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"   Using device: {computation_device}")    
case_outcome_model = create_model_from_data(court_data, get_cache_dir('bert'))
analyze_model_memory(case_outcome_model)

In [ ]:
print_label_distribution_by_court(
    court_data,
    cached_outcome_labels_v2,
    model_configuration.LABEL_NAMES,
    mag_key="mag",
    cdc_key="cdc",
    proceeding_col="proceeding_id")

## 2. Instantiate model

In [ ]:
computation_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_cache_directory = get_cache_dir('bert')
proceeding_index_file = bert_cache_directory / "proceeding_index.pkl"
# only build the index (and embeddings.h5) if it doesn't exist
print(f"Checking for cached BERT embeddings in {bert_cache_directory}")
if not proceeding_index_file.exists():
   extract_and_cache_bert_embeddings_streaming(
        court_data,
        cache_dir=bert_cache_directory,
        model_name="bert-base-uncased",
        out_dim=768,
        batch_size=32,
        device=computation_device,
        force_recompute=False
    )

In [ ]:
print("Creating model from data")
case_outcome_model = create_model_from_data(court_data, bert_cache_directory)

## 3. Build list of (mag_num, case_id, defendant_id) tuples

In [ ]:
print("Building list of defendant cases for training")
combined_dockets_df = court_data['combined_dockets']
defendant_case_identifiers = []
for _, row in combined_dockets_df.iterrows():
    case_identifier = (row.get('mag_num'), row.get('case_id'), row.get('defendant_id'))
    defendant_case_identifiers.append(case_identifier)
valid_defendant_case_ids = [
    case_id for case_id in defendant_case_identifiers 
    if pd.notna(case_id[2]) and (pd.notna(case_id[0]) or pd.notna(case_id[1]))
]

## 4. Sample a subset and create sequences with prior-case features


In [ ]:
print("Filtering to completed cases only for training")
completed_defendant_cases = filter_to_completed_cases(court_data, valid_defendant_case_ids, cached_outcome_labels_v2)
print(f"Found {len(completed_defendant_cases)} completed cases for training")

In [ ]:
# Pre-filter valid cases first
print("Pre-filtering valid cases...")
_counts = build_case_counts_df(court_data, cached_outcome_labels_v2)

_df_cases = pd.DataFrame(completed_defendant_cases,
                         columns=['mag_num','case_id','defendant_id'])

_tagged = (_df_cases.merge(_counts, on=['mag_num','case_id','defendant_id'], how='left')
                    .assign(valid=lambda d: d['valid'].fillna(False)))

valid_completed_defendant_cases = [tuple(x) for x in
    _tagged.loc[_tagged['valid'], ['mag_num','case_id','defendant_id']].to_numpy()]

In [ ]:
if not parameters_dict['new_dataset_creation']:
    print(f"Sampling {min(parameters_dict['number_of_cases_in_sample'], len(valid_completed_defendant_cases))} out of {len(valid_completed_defendant_cases)} valid cases")
    training_case_sample = random.sample(valid_completed_defendant_cases, min(parameters_dict['number_of_cases_in_sample'], len(valid_completed_defendant_cases)))
else: 
    training_case_sample = valid_completed_defendant_cases

In [ ]:
models_directory_cooccurences_path = get_cache_dir('models')
models_directory_cooccurences_path.mkdir(parents=True, exist_ok=True)
cooccurrences_output_file = models_directory_cooccurences_path / "cooccurrence_matrix_valid_completed_cases.xlsx"

In [ ]:
export_cooccurrence_from_valid_completed_cases(
    valid_completed_defendant_cases=valid_completed_defendant_cases,
    court_data=court_data,
    cached_outcome_labels_v2=cached_outcome_labels_v2,
    label_names=model_configuration.LABEL_NAMES,
    output_path=cooccurrences_output_file
)

In [ ]:
clerk_cache_dir = get_cache_dir('clerk_analysis')
analyze_clerk_data(
    court_data,
    cached_outcome_labels_v2,
    valid_completed_defendant_cases, 
    output_dir = clerk_cache_dir
)

In [ ]:
print("Creating Enhanced Label Processor with predefined labels")
# Initialize the label processor with your 22 predefined labels
label_processor = EnhancedLabelProcessor(predefined_label_names=model_configuration.LABEL_NAMES)

print("Creating training sequences with MultiLabelBinarizer support")
# Use the new sequence creation function that returns raw string labels
sequences, label_sequences_raw = create_enhanced_training_sequences_cached(
    training_case_sample, 
    court_data, 
    cached_outcome_labels_v2
)

print(f"Created {len(sequences)} sequences")
print(f"Sample sequence length: {len(sequences[0]) if sequences else 0}")
print(f"Sample labels: {label_sequences_raw[0][0] if label_sequences_raw else []}")

# Fit the MultiLabelBinarizer on all your label data
print("Fitting MultiLabelBinarizer on training data...")
label_processor.fit(label_sequences_raw)

# Verify the fitted labels match your expectations
print(f"Label processor fitted with {len(label_processor.label_names)} labels:")
print(f"Expected: {len(model_configuration.LABEL_NAMES)} labels")
print(f"Match: {set(label_processor.label_names) == set(model_configuration.LABEL_NAMES)}")

In [ ]:
if parameters_dict['new_dataset_creation']:
    cases = list(zip(sequences, label_sequences_raw))
    sampled_cases, sampled_lengths, sampled_indices = sample_from_the_longest_cases(
    cases,
    number_of_longest_cases=5000,
    sample_size=1000,
    random_seed=42)

    export_long_cases_to_xlsx(sampled_cases, output_filepath=get_cache_dir('datasets') / "longest_case_samples.xlsx")
    sequences_v2, label_sequences_raw_v2 = zip(*sampled_cases)
    sequences = list(sequences_v2)
    label_sequences_raw = list(label_sequences_raw_v2)

In [ ]:
# save the label processor for future use
with open(get_cache_dir('models') / 'label_processor.pkl', 'wb') as file:
    pickle.dump(label_processor, file)
#save the sequences and label_sequences_raw for future use
with open(get_cache_dir('datasets') / 'sequences_and_labels_old.pkl', 'wb') as file:
    pickle.dump((sequences, label_sequences_raw), file)
with open(get_cache_dir('datasets') / 'sequences_and_labels.pkl', 'wb') as file:
    pickle.dump((sequences_v2, label_sequences_raw_v2), file)
with open(get_cache_dir('datasets') / 'sampled_indices.pkl', 'wb') as file:
    pickle.dump(sampled_indices, file)

In [ ]:
#open the label processor
with open(get_cache_dir('models') / 'label_processor.pkl', 'rb') as file:
    label_processor = pickle.load(file)
#open the sequences and label_sequences_raw
with open(get_cache_dir('datasets') / 'sequences_and_labels.pkl', 'rb') as file:
    sequences, label_sequences_raw = pickle.load(file)
    